# **MovRec: Sistem Rekomendasi Film Netflix Berbasis Mood Kategori dengan Fitur Random, Genre, dan Movie of The Day**

**Kelompok Data Science Beginner**


1.   Nayla Dwinta Putri Muharram (2410512017)
2.   Lathisya Sheza Aldamar (2410512006)
3.   Regina Juliyanti Manalu (2410512016)



##**1. Import Library**

In [1]:
# Data Processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text Processing
import re
import string

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Random Recommendation
import random

# Ignore Warning
import warnings
warnings.filterwarnings('ignore')

##**2. Data Collecting**

2.1. Connect Gdrive dan Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/netflix_titles.csv')

2.2. Menampilkan 5 data awal

In [ ]:
df.head()

2.3. Menampilkan 5 Data terakhir

In [ ]:
df.tail()

2.4. Menampilkan jumlah baris dan kolom

In [ ]:
df.shape

2.5. Menampilkan nama-nama kolom

In [ ]:
df.columns

## **3. Exploratory Data Analysis**

3.1. Rangkuman Informasi Dataset

In [ ]:
df.info()

3.2. Cek Missing Value per Kolom

In [ ]:
df.isnull().sum()

3.3. Melihat distribusi tipe konten Netflix (Movie vs TV Show)

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(x='type', data=df)

plt.title('Distribusi Movie dan TV Show')

plt.show()

3.4. Mengetahui top 10 genre Netflix

In [ ]:
genre_count = df['listed_in'].str.split(', ', expand=True).stack().value_counts().head(10)

plt.figure(figsize=(10,5))

genre_count.plot(kind='bar')

plt.title('Top 10 Genre Netflix')

plt.xlabel('Genre')

plt.ylabel('Jumlah')

plt.show()

3.5. Mengetahui negara dengan produksi film terbanyak

In [ ]:
country_count = df['country'].value_counts().head(10)

plt.figure(figsize=(10,5))

country_count.plot(kind='bar')

plt.title('Top Negara Produksi')

plt.show()

3.6. Melihat tren perilisan film Netflix

In [ ]:
plt.figure(figsize=(10,5))

df['release_year'].value_counts().sort_index().plot()

plt.title('Distribusi Tahun Rilis Film')

plt.xlabel('Tahun')

plt.ylabel('Jumlah')

plt.show()

## **4. Data Preprocessing**

4.1. Mengisi Missing Value

In [ ]:
df['director'] = df['director'].fillna('')
df['cast'] = df['cast'].fillna('')
df['country'] = df['country'].fillna('')
df['description'] = df['description'].fillna('')
df['listed_in'] = df['listed_in'].fillna('')

# Karena dalam machine learning dan NLP, model tidak bisa memproses data NaN. Jadi missing value diisi supaya:
# proses preprocessing tidak error
# TF-IDF bisa berjalan
# fitur gabungan (combined_features) tidak gagal dibuat

4.2. Menghapus Data Duplikat

In [ ]:
df.drop_duplicates(inplace=True)

4.3. Membersihkan text agar lebih mudah diproses oleh model NLP

In [ ]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'\[.*?\]', '', text)

    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    text = re.sub(r'<.*?>+', '', text)

    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)

    text = re.sub(r'\n', '', text)

    text = re.sub(r'\w*\d\w*', '', text)

    return text

4.4. Membersihkan Kolom Description

In [ ]:
df['clean_description'] = df['description'].apply(clean_text)

4.5. Menggabungkan beberapa fitur penting menjadi satu text gabungan

In [ ]:
df['combined_features'] = (
    df['clean_description'] + ' ' +
    df['listed_in'] + ' ' +
    df['cast'] + ' ' +
    df['director']
)

Hasil Gabungan Fitur Engineering

In [ ]:
df[['title', 'combined_features']].head()

## **5. Model Training**

5.1. TF-IDF Vectorization

In [ ]:
# Mengubah data text menjadi bentuk numerik

tfidf = TfidfVectorizer(stop_words='english')

5.1.2. Fit dan Transform Data

In [ ]:
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

5.1.3 Shape Matrix

In [ ]:
tfidf_matrix.shape

5.2. Cosine Similarity

In [ ]:
# Menghitung tingkat kemiripan antarfilm

cosine_sim = cosine_similarity(tfidf_matrix)

5.3. Membuat Index Film

In [ ]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

5.4. Memberikan rekomendasi berdasarkan judul film

In [ ]:
def recommend_movies(title):

    if title not in indices:

        return "Movie not found in dataset"

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores,
                        key=lambda x: x[1],
                        reverse=True)

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return df[['title', 'listed_in']].iloc[movie_indices]

5.5. Memberikan rekomendasi film berdasarkan kategori mood yang dipilih user

In [ ]:
mood_keywords = {

    "Happy": "comedy family",

    "Sad": "drama emotional",

    "Excited": "action thriller",

    "Romantic": "romance love",

    "Sci-Fi": "science fiction space",

    "Chill": "documentary drama"
}

5.6. Function Recommendation by Mood

In [ ]:
def recommend_by_mood(mood):

    keywords = mood_keywords[mood]

    user_vector = tfidf.transform([keywords])

    similarity = cosine_similarity(user_vector, tfidf_matrix)

    similarity_scores = list(enumerate(similarity[0]))

    similarity_scores = sorted(similarity_scores,
                               key=lambda x: x[1],
                               reverse=True)

    top_movies = similarity_scores[:10]

    movie_indices = [i[0] for i in top_movies]

    return df[['title', 'listed_in', 'description']].iloc[movie_indices]

5.7. Random Movie Picker

In [ ]:
def random_movie():

    random_pick = df.sample(1)

    return random_pick[['title', 'listed_in', 'description']]

5.8. Top 10 Action Movies

In [ ]:
action_movies = df[df['listed_in'].str.contains('Action', case=False)]

top_action_movies = action_movies[['title', 'listed_in']].head(10)

top_action_movies

5.9. Movie of The Day

In [ ]:
movie_of_the_day = df.sample(1)

movie_of_the_day[['title', 'description']]

## **6. Model Evaluation**

**6.1. Testing Recommendation by Title**

In [ ]:
recommend_movies('9')

**6.2. Testing Recommendation by User's Select Mood**

In [ ]:
recommend_by_mood('Happy')

**6.3. Analisis Hasil Rekomendasi**

Berdasarkan hasil pengujian, sistem mampu memberikan rekomendasi film yang sesuai dengan mood yang dipilih pengguna

Sebagai contoh, ketika pengguna memilih mood 'Happy', sistem merekomendasikan film dengan genre comedy, family, dan feel-good movie yang memiliki kemiripan fitur berdasarkan perhitungan *cosine similarity*

**6.4. Evaluasi Similiarity Score**

In [ ]:
selected_mood = 'Happy'

keywords = mood_keywords[selected_mood]

user_vector = tfidf.transform([keywords])

similarity = cosine_similarity(user_vector, tfidf_matrix)

similarity_scores = list(enumerate(similarity[0]))

similarity_scores = sorted(similarity_scores,
                           key=lambda x: x[1],
                           reverse=True)

similarity_scores[:10]

In [ ]:
top_movies = similarity_scores[:10]

# Hasil Evaluasi di atas masih berbentuk index, sehingga diubah dahulu menjadi judul

for i in top_movies:

    movie_index = i[0]

    score = i[1]

    print(
        "Title:",
        df.iloc[movie_index]['title']
    )

    print(
        "Similarity Score:",
        round(score, 3)
    )

    print("-" * 40)

**6.5. Analisis Hasil Similiarity Score**

Berdasarkan hasil evaluasi *similarity score*, sistem berhasil menemukan film yang memiliki kemiripan dengan mood 'Happy'.

Film yang direkomendasikan umumnya memiliki genre *comedy, family*, dan *entertainment* yang sesuai dengan keyword mood yang digunakan.

Semakin tinggi *similarity score*, maka semakin relevan film tersebut terhadap mood pengguna.

**6.6. Kesimpulan Evaluasi**

Model rekomendasi sistem berbasis TF-IDF dan *cosine similarity* berhasil memberikan rekomendasi film berdasarkan kategori mood yang dipilih pengguna.

Hasil skor sebanding menunjukkan bahwa sistem dapat menemukan film dengan genre dan deskripsi yang sesuai dengan mood seperti bahagia, sedih, romantis, dan *excited*.

In [ ]:
!pip install joblib
import joblib

In [ ]:
joblib.dump(df, 'movies.pkl')
joblib.dump(tfidf, 'tfidf.pkl')
joblib.dump(tfidf_matrix, 'tfidf_matrix.pkl')

In [ ]:
from google.colab import files

files.download('movies.pkl')
files.download('tfidf.pkl')
files.download('tfidf_matrix.pkl')